# Gemma 4 E4B Jordan Peterson Fine-Tuning
## Google Gemma 4 (Effective 4B) · V4 Q&A Data · 3 Epochs · r=16 LoRA

This notebook fine-tunes **Gemma 4 E4B** on Jordan B. Peterson's four books using
the same V4 training pipeline as `Qwen3_14B_JordanPeterson_V4_FineTuning.ipynb`.

**Pipeline position:** Run `JordanPeterson_DataPrep.ipynb` first to generate
the Q&A cache (`qa_dataset/peterson_qa.jsonl`).  This notebook reads from that
cache directly — no PDF extraction or question generation happens here.

### Why Gemma 4 E4B?

Gemma 4 is Google's April 2026 model family.  E4B uses a Mixture-of-Experts (MoE)
architecture with 4B *effective* active parameters (drawn from a larger total
parameter count), giving strong quality at low inference cost.

| Property | Value |
|----------|-------|
| Model | `unsloth/gemma-4-E4B-it` |
| Architecture | MoE — 4B effective active params |
| VRAM (LoRA training) | ~17 GB |
| RTX 4090 | 24 GB — comfortable headroom |
| Chat format | `<start_of_turn>user/model` |
| System prompt support | Yes (prepended to first user turn) |

---
## Step 0: Upgrade Unsloth and Transformers

Gemma 4 requires:
- **Unsloth ≥ 2026.4.x** — adds `gemma-4` chat template and model support
- **Transformers ≥ 5.5.0** — adds the `Gemma4` model class

Run the cell below, then **restart the kernel** before continuing.

In [1]:
# this was already done, so let's skip
if False:
    # Upgrade Unsloth and Transformers for Gemma 4 support.
    # After this cell completes, restart the kernel before running further cells.
    import subprocess, sys

    packages = ["unsloth", "unsloth-zoo", "transformers>=5.5.0"]
    print("Upgrading:", packages)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade"] + packages,
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else "(no stdout)")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
    else:
        print("\nUpgrade complete.  Restart the kernel, then run the remaining cells.")

---
## Imports and Configuration

In [2]:
import json, math, time, gc
from pathlib import Path

import torch
from datasets import Dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# ── Load shared config ─────────────────────────────────────────────
with open("peterson_config.json") as f:
    _config = json.load(f)

QA_CACHE      = Path(_config["paths"]["qa_cache"])
SYSTEM_PROMPT = _config["system_prompt"]

# ── Model + training constants ───────────────────────────────────────────
BASE_MODEL    = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LEN   = 2048
LORA_RANK     = 32          # r=32 matches Qwen3-14B V4; comfortable on 24 GB with E4B weights
LORA_ALPHA    = 32          # alpha/rank = 1.0
BATCH_SIZE    = 2           # E4B weights only ~10 GB; 14 GB headroom allows batch=2
GRAD_ACCUM    = 4           # effective batch = 2 × 4 = 8 (same as before)
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-4
OUTPUT_DIR    = Path("outputs/gemma4_e4b_peterson_v1_lora")

_total_steps  = math.ceil(len(list(open(QA_CACHE))) / BATCH_SIZE) * NUM_EPOCHS // GRAD_ACCUM
print(f"Config loaded.  Q&A cache: {QA_CACHE}")
print(f"Base model    : {BASE_MODEL}")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"LoRA rank     : r={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"Training      : {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, accum={GRAD_ACCUM}")
print(f"~{_total_steps:,} gradient updates expected")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Config loaded.  Q&A cache: qa_dataset/peterson_qa.jsonl
Base model    : unsloth/gemma-4-E4B-it
Output dir    : outputs/gemma4_e4b_peterson_v1_lora
LoRA rank     : r=32, alpha=32
Training      : 3 epochs, batch=2, accum=4
~1,885 gradient updates expected


---
# Part 1: Load the Q&A Dataset

Same V4 cache used by all recent fine-tuning notebooks:
**3,936 pairs** from **1,968 passages** with both front-matter AND back-matter removed.

In [3]:
from collections import Counter

if not QA_CACHE.exists():
    raise FileNotFoundError(
        f"Q&A cache not found at {QA_CACHE}.\n"
        "Run JordanPeterson_DataPrep.ipynb first to generate the dataset."
    )

with open(QA_CACHE) as f:
    records = [json.loads(line) for line in f if line.strip()]

raw_dataset = Dataset.from_list([
    {"question": r["question"], "answer": r["answer"]}
    for r in records
])

print(f"Dataset loaded: {len(raw_dataset):,} Q&A pairs")
print(f"Schema: {raw_dataset.column_names}")
print()

book_dist = Counter(r["book"] for r in records)
print("Distribution by book:")
for book, count in sorted(book_dist.items(), key=lambda x: -x[1]):
    print(f"  {book:<35} {count:>5} pairs")

Dataset loaded: 5,028 Q&A pairs
Schema: ['question', 'answer']

Distribution by book:
  We Who Wrestle with God              2330 pairs
  Maps of Meaning                      1042 pairs
  12 Rules for Life                     878 pairs
  Beyond Order                          778 pairs


---
# Part 2: Load the Gemma 4 E4B Model

## Architecture: Mixture-of-Experts with Effective 4B Active Parameters

Gemma 4 E4B uses a sparse MoE design where each forward pass activates only a
4B-parameter subset of the full model, giving:

- **Low inference cost** — only 4B params computed per token
- **High capacity** — total parameter count is much larger
- **Strong quality** — MoE models punch above their active-parameter weight

## Loading via `FastModel`

Gemma 4 uses the unified `FastModel` API (not `FastLanguageModel` or
`FastVisionModel`), which handles both text-only and multimodal variants.

In [4]:
print(f"Loading {BASE_MODEL} ...")

model, tokenizer = FastModel.from_pretrained(
    model_name      = BASE_MODEL,
    dtype           = None,          # auto: bfloat16 on Ampere+
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,          # 4-bit quantization via bitsandbytes
    full_finetuning = False,         # LoRA adapters only
)

vram_after_load = torch.cuda.memory_reserved() / 1e9
print(f"Model loaded.  VRAM reserved : {vram_after_load:.1f} GB")
print(f"Model dtype   : {next(model.parameters()).dtype}")
print(f"Tokenizer type: {type(tokenizer).__name__}")
free_vram = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_reserved()) / 1e9
print(f"Free VRAM     : {free_vram:.1f} GB")

Loading unsloth/gemma-4-E4B-it ...
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.635 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Model loaded.  VRAM reserved : 10.0 GB
Model dtype   : torch.bfloat16
Tokenizer type: Gemma4Processor
Free VRAM     : 15.3 GB


---
# Part 3: Add LoRA Adapters (r=16)

## Why r=16 Instead of r=32?

The E4B model has 4B *effective* active parameters per forward pass.  Compared to
the Qwen3-14B (14B dense parameters), a lower rank is appropriate:

- Proportionally r=16 on E4B ≈ r=32 on 14B (similar fraction of model capacity)
- Keeps the adapter checkpoint small (~130 MB vs ~260 MB for r=32)
- Reduces risk of overfitting on the same 3,936-pair dataset

## Gemma 4 LoRA API

`FastModel.get_peft_model` uses a declarative API for multimodal models.
Since we are doing text-only fine-tuning, we disable the vision layers.

In [5]:
model = FastModel.get_peft_model(
    model,
    r                       = LORA_RANK,
    lora_alpha              = LORA_ALPHA,
    lora_dropout            = 0,             # 0 is optimal for Unsloth
    finetune_vision_layers  = False,         # skip vision encoder (text-only task)
    finetune_language_layers= True,          # LoRA on all text/LM layers
    use_gradient_checkpointing = "unsloth",  # Unsloth optimised checkpointing
    random_state            = 42,
    bias                    = "none",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.2f}% of total)")
print(f"Total params     : {total:,}")

Trainable params : 73,400,320  (1.21% of total)
Total params     : 6,052,686,368


---
# Part 4: Format Dataset into Gemma 4 Chat Format

## Gemma 4 Chat Template

Gemma 4 uses the same `<start_of_turn>` / `<end_of_turn>` token format as Gemma 3:

```
<bos><start_of_turn>user
{SYSTEM_PROMPT}

{question}<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>
<eos>
```

Note: The system message is **prepended to the first user turn** (not given its own
`<start_of_turn>system` wrapper).  This is handled automatically by the tokenizer's
built-in chat template — we just pass `role: "system"` as normal.

## `train_on_responses_only`

We mask the user/system portion of each training sequence and only compute loss on
the model's response.  The response boundary is `<start_of_turn>model\n`.

In [6]:
def format_example(batch):
    '''Convert Q&A records into Gemma 4 chat format strings.'''
    formatted_texts = []
    for question, answer in zip(batch["question"], batch["answer"]):
        conversation = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ]
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize              = False,
            add_generation_prompt = False,
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = raw_dataset.map(format_example, batched=True)
print(f"Dataset formatted: {len(dataset):,} examples")
print()
print("Sample formatted text (first 600 chars):")
print(dataset[0]["text"][:600])

Map:   0%|          | 0/5028 [00:00<?, ? examples/s]

Dataset formatted: 5,028 examples

Sample formatted text (first 600 chars):
<bos><|turn>system
You are an AI assistant that has been trained on the complete works of Jordan B. Peterson, a Canadian clinical psychologist, professor, and author. You speak with deep knowledge of psychology, philosophy, mythology, religion, and personal responsibility. Your responses reflect Peterson's writing style, intellectual depth, and interdisciplinary approach to understanding human nature and meaning.<turn|>
<|turn>user
How does the brain's dual-hemisphere structure relate to the mythological opposition between order and chaos?<turn|>
<|turn>model
FIGURES 1 The Domain and Constitue


In [7]:
# Auto-detect Gemma 4 response boundary tokens.
# Gemma 4 (April 2026 tokenizer) uses <|turn>model as the response boundary.
_sample_text = dataset[0]["text"]

if "<|turn>model\n" in _sample_text:
    instruction_part = "<|turn>user\n"
    response_part    = "<|turn>model\n"
    print("Detected Gemma 4 response boundary tokens:")
elif "<start_of_turn>model\n" in _sample_text:
    instruction_part = "<start_of_turn>user\n"
    response_part    = "<start_of_turn>model\n"
    print("Detected Gemma 3-style response boundary tokens:")
else:
    raise RuntimeError(
        "Could not detect Gemma 4 chat tokens in formatted dataset.\n"
        "Expected '<|turn>model\\n' or '<start_of_turn>model\\n' — check tokenizer output above."
    )

print(f"  instruction_part : {repr(instruction_part)}")
print(f"  response_part    : {repr(response_part)}")

sft_trainer_temp = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    args             = SFTConfig(
        dataset_text_field = "text",
        max_length         = MAX_SEQ_LEN,
        output_dir         = "/tmp/gemma4_tok_check",
    ),
)
sft_trainer_temp = train_on_responses_only(
    sft_trainer_temp,
    instruction_part = instruction_part,
    response_part    = response_part,
)
print("train_on_responses_only applied successfully.")

Detected Gemma 4 response boundary tokens:
  instruction_part : '<|turn>user\n'
  response_part    : '<|turn>model\n'


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

Filter (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

train_on_responses_only applied successfully.


---
# Part 5: Configure and Run Training

## VRAM Budget on RTX 4090 (24 GB)

| Component | Estimate |
|-----------|----------|
| Gemma 4 E4B (4-bit) | ~8–10 GB |
| LoRA adapters (r=16) | ~0.5 GB |
| Activations + grad checkpointing | ~4–6 GB |
| 8-bit Adam optimizer states | ~0.5 GB |
| **Total** | **~14–17 GB** (7–10 GB headroom) |

## Key Parameter Choices

- **`batch_size=1`** — MoE models route tokens through multiple experts, making
  activations harder to predict; start conservative
- **`grad_accum=8`** — effective batch = 8 (same as all other notebooks)
- **`learning_rate=2e-4`** — standard LoRA LR; reduce to `2e-5` if loss diverges
- **`warmup_steps=30`** — same ramp as previous notebooks

In [8]:
import os
os.environ["NCCL_P2P_DISABLE"] = "1"   # RTX 4000-series P2P fix
os.environ["NCCL_IB_DISABLE"]  = "1"

sft_trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    args             = SFTConfig(
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ_LEN,
        dataset_num_proc            = 2,

        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = NUM_EPOCHS,

        learning_rate               = LEARNING_RATE,
        warmup_steps                = 30,
        lr_scheduler_type           = "linear",
        optim                       = "adamw_8bit",

        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),

        logging_steps               = 25,
        output_dir                  = str(OUTPUT_DIR / "checkpoints"),
        save_strategy               = "no",
        report_to                   = "none",
    ),
)
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = instruction_part,
    response_part    = response_part,
)
print(f"SFTTrainer ready.  Expected ~{_total_steps:,} gradient updates.")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5028 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

Filter (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

SFTTrainer ready.  Expected ~1,885 gradient updates.


In [9]:
vram_before = torch.cuda.memory_reserved() / 1e9
print(f"VRAM before training : {vram_before:.1f} GB")
print(f"Starting training — {NUM_EPOCHS} epochs, ~{_total_steps:,} gradient updates...")
print()

t0 = time.time()

try:
    train_result = sft_trainer.train()
except RuntimeError as e:
    if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
        vram_peak = torch.cuda.max_memory_reserved() / 1e9
        print(f"\n✗  OOM during training!  Peak VRAM: {vram_peak:.1f} GB")
        print("\nFallback options:")
        print("  1. Reduce BATCH_SIZE to 1 and GRAD_ACCUM to 4 (already at minimum)")
        print("  2. Reduce MAX_SEQ_LEN from 2048 to 1024")
        print("  3. Reduce LORA_RANK from 16 to 8")
        raise
    raise

elapsed_min  = (time.time() - t0) / 60
vram_peak    = torch.cuda.max_memory_reserved() / 1e9

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Steps          : {train_result.global_step:,}")
print(f"  Elapsed        : {elapsed_min:.1f} min")
print(f"  Final loss     : {train_result.training_loss:.4f}")
print(f"  Peak VRAM      : {vram_peak:.1f} GB")
print()
print("Note: E2B/E4B MoE models normally show loss 13–15 (expected for multimodal")
print("  MoE architectures).  The 26B/31B variants show loss 1–3 instead.")
print("  A loss of ~13–15 here is NOT a sign that training failed.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


VRAM before training : 10.3 GB
Starting training — 3 epochs, ~1,885 gradient updates...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,028 | Num Epochs = 3 | Total steps = 1,887
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 73,400,320 of 8,069,556,768 (0.91% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,7.312841
50,4.010285
75,3.558542
100,3.416300
125,3.239666
150,3.295599
175,3.188996
200,3.156506
225,3.074272
250,3.089783



TRAINING COMPLETE
  Steps          : 1,887
  Elapsed        : 64.3 min
  Final loss     : 2.2169
  Peak VRAM      : 13.3 GB

Note: E2B/E4B MoE models normally show loss 13–15 (expected for multimodal
  MoE architectures).  The 26B/31B variants show loss 1–3 instead.
  A loss of ~13–15 here is NOT a sign that training failed.


---
# Part 6: Save the LoRA Adapter

We save only the LoRA adapter weights — not the full model.  The checkpoint will
be small (~130 MB for r=16) compared to the base model size.

**Output path**: `outputs/gemma4_e4b_peterson_v1_lora/`

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving LoRA adapter to {OUTPUT_DIR} ...")

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

adapter_files = (
    list(OUTPUT_DIR.glob("*.safetensors"))
    + list(OUTPUT_DIR.glob("*.bin"))
)
total_mb = sum(f.stat().st_size for f in adapter_files) / 1e6

print()
print("Adapter files:")
for f in sorted(adapter_files):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
print()
print(f"Total adapter size : {total_mb:.1f} MB")
print(f"  (Qwen3-14B V4 r=32 was 513.9 MB; Gemma 4 E4B r=16 should be ~130 MB)")

Saving LoRA adapter to outputs/gemma4_e4b_peterson_v1_lora ...

Adapter files:
  adapter_model.safetensors  (293.7 MB)

Total adapter size : 293.7 MB
  (Qwen3-14B V4 r=32 was 513.9 MB; Gemma 4 E4B r=16 should be ~130 MB)


---
# Part 7: Inference Test

Same 5 evaluation prompts used across all fine-tuning notebooks for direct
comparison.  Greedy decoding (`do_sample=False`) for deterministic, reproducible
outputs.

In [12]:
FastModel.for_inference(model)

EVAL_PROMPTS = [
    "What is the relationship between order and chaos in human experience?",
    "Why is personal responsibility the foundation of a meaningful life?",
    "How do ancient myths and stories reveal truths about human nature?",
    "What does it mean to pursue what is meaningful rather than what is expedient?",
    "How should a person confront suffering rather than flee from it?",
]


def ask(question: str, max_new_tokens: int = 300) -> str:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "text", "text": question}]},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens = max_new_tokens,
            do_sample      = False,   # greedy — deterministic for comparison
            use_cache      = True,
        )

    new_tokens = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


for i, prompt in enumerate(EVAL_PROMPTS, start=1):
    print(f"{'='*60}")
    print(f"Prompt {i}: {prompt}")
    print(f"{'-'*60}")
    response = ask(prompt)
    print(response)
    print()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt 1: What is the relationship between order and chaos in human experience?
------------------------------------------------------------
the world as it is, and how it could be. The world as it is—that is the world of experience, as it manifests itself to you. It is the world of your emotions, your thoughts, your actions, and the actions of others. It is the world of your past, present, and future. It is the world of your deepest being, and it is not limited by what you know, or what you think you know. It is the totality of what is, and what could be. It is the world of your becoming, and it is not limited by what you have already made. It is the world of your becoming, and it is not limited by what you have said. It is the world of your becoming, and it is not limited by what you have done. It is the world of your becoming, and it is not limited by what you will do. It is the world of your becoming, and it is not limited by what you could do. It is the world of your becoming, and

---
# Conclusions

## What This Notebook Tests

This is the first Gemma 4 fine-tuning experiment in this repository.  It tests
whether Google's MoE architecture (effective 4B active params) can absorb
Peterson's philosophical style as effectively as the Qwen3 family.

## Key Comparisons

| Aspect | Qwen3-14B V4 | **Gemma 4 E4B** |
|--------|-------------|----------------|
| Parameters (active) | 14B (dense) | ~4B (MoE) |
| Architecture | Dense transformer | Mixture-of-Experts |
| Training data | 3,936 Q&A pairs | 3,936 Q&A pairs (same) |
| LoRA rank | r=32 | r=16 |
| Effective batch | 8 | 8 |
| Epochs | 3 | 3 |
| Chat format | Qwen3 ChatML | Gemma `<start_of_turn>` |

## Expected Loss

Unsloth documents that E2B/E4B models show **loss 13–15** during training, compared
to 1–3 for the 26B/31B variants.  This is a known characteristic of smaller MoE
models and does NOT indicate poor training quality — the model still learns the
fine-tuning signal, just with higher baseline perplexity on the raw token distribution.

## Next Steps

- Run `AllModels_JordanPeterson_Comparison.ipynb` (update it to add `gemma4_e4b`)
- Compare perplexity, TF-IDF similarity, and keyword density vs Qwen3-14B V4
- Consider Gemma 4 26B-A4B if access to a multi-GPU setup becomes available
  (requires >40 GB VRAM total)

In [13]:
!nvidia-smi

Fri Apr 10 13:06:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:2B:00.0  On |                  Off |
|  0%   40C    P8             20W /  450W |   11224MiB /  24564MiB |     33%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----